# Notebook: Pandas for Historians

**Objective:** Move beyond "looking" at data in Excel to "interrogating" it with Python.  
**Scenario:** You have digitized a set of ship manifests from 1845–1855. You want to understand migration patterns, but the data is messy, incomplete, and too large to count by hand.

## 1. Setup & Creating Our "Archive"
First, we import `pandas`. Since we don't have a real archive handy on your local machine yet, we will run a quick script to generate a "dummy" historical dataset.

*(In your real research, you would skip the data generation step and just load your own `.csv` file.)*

In [1]:
import pandas as pd
import numpy as np

# --- TEACHING SETUP: Generating a fake historical dataset ---
data = {
    'PassengerID': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    'Surname': ['O\'Malley', 'Schmidt', 'Romano', 'O\'Malley', 'Dubois', 'Unknown', 'Müller', 'Ivanov', 'Kelly', 'Chen', 'Smith', 'O\'Connor'],
    'Given Name': ['Patrick', 'Hans', 'Giuseppe', 'Mary', 'Jean', np.nan, 'Wilhelm', 'Dmitri', 'Sean', 'Wei', 'John', 'Bridget'],
    'Age': [24, 35, 19, 22, 40, np.nan, 29, 31, 18, 26, 45, 20],
    'Occupation': ['Laborer', 'Baker', 'Laborer', 'Seamstress', 'Merchant', np.nan, 'Baker', 'Soldier', 'Laborer', 'Merchant', 'Clerk', 'Servant'],
    'Origin': ['Cork', 'Hamburg', 'Naples', 'Cork', 'Lyon', 'Unknown', 'Hamburg', 'Odessa', 'Dublin', 'Canton', 'London', 'Cork'],
    'Arrival Date': ['1845-04-12', '1848-06-20', '1849-11-02', '1845-04-12', '1850-01-15', '1847-09-10', '1848-06-20', '1852-03-22', '1845-05-05', '1854-07-30', '1855-01-01', '1845-04-12']
}

# Create the CSV file locally
df_setup = pd.DataFrame(data)
df_setup.to_csv('ship_manifest_1845_1855.csv', index=False)

print("Dataset 'ship_manifest_1845_1855.csv' has been created in your project folder.")

Dataset 'ship_manifest_1845_1855.csv' has been created in your project folder.


## 2. The "Load" (Reading the Source)
In traditional history, you request a box from the archivist. In Python, you read a CSV.

In [2]:
# Load the dataset
df = pd.read_csv('ship_manifest_1845_1855.csv')

# Inspect the first 5 rows (The "Quick Look")
df.head()

,PassengerID,Surname,Given Name,Age,Occupation,Origin,Arrival Date
0,101,O'Malley,Patrick,24.0,Laborer,Cork,1845-04-12
1,102,Schmidt,Hans,35.0,Baker,Hamburg,1848-06-20
2,103,Romano,Giuseppe,19.0,Laborer,Naples,1849-11-02
3,104,O'Malley,Mary,22.0,Seamstress,Cork,1845-04-12
4,105,Dubois,Jean,40.0,Merchant,Lyon,1850-01-15


**Observation:** Look at the data above. Notice that Python has successfully loaded the table, but we have some obvious issues—`NaN` values (missing data) and special characters in names.

## 3. The Inspection (Metadata)
Before analyzing, we must understand the "shape" of our archive.
* **`df.info()`** tells us about data types (Is the date treated as a number? Is age text?).
* **`df.describe()`** gives us summary statistics (What is the average age?).

In [3]:
# Check data types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerID   12 non-null     int64  
 1   Surname       12 non-null     str    
 2   Given Name    11 non-null     str    
 3   Age           11 non-null     float64
 4   Occupation    11 non-null     str    
 5   Origin        12 non-null     str    
 6   Arrival Date  12 non-null     str    
dtypes: float64(1), int64(1), str(5)
memory usage: 804.0 bytes


* **Critique:** Look at the `Arrival Date` column in the output above. It is listed as an `object` (string). This is bad! We cannot graph a timeline if Python thinks the date is just text. We will fix this later.

In [ ]:
# Get basic statistics for numerical columns
df.describe()

**Historical Question:** Does the "Mean" age (approx. 28 years old) make sense for a dataset of immigrants? What does the "Min" (18) and "Max" (45) tell us about this specific cohort?

## 4. Data Cleaning (Handling Silence)
Historical data is full of "silence"—gaps, torn pages, and illegible handwriting. In Pandas, these are represented as `NaN` (Not a Number) or `null`.

We have a choice:
1.  **Drop** the incomplete records (Risk: erasing history).
2.  **Fill** them with placeholders (Risk: inventing data).

Let's see where our silence is:

In [4]:
# Identify which rows have missing data
# Returns True if data is missing, False if it exists
missing_data = df.isnull().sum()
print(missing_data)

PassengerID     0
Surname         0
Given Name      1
Age             1
Occupation      1
Origin          0
Arrival Date    0
dtype: int64


For this project, let's say we are only interested in passengers whose `Age` is known. We will filter out the empty rows.

In [ ]:
# Drop rows where 'Age' is NaN (Not a Number)
# inplace=True means "modify the original dataframe, don't just make a copy"
df.dropna(subset=['Age'], inplace=True)

# Verify the count
len(df)

## 5. Filtering (The "Where" Clause)
A historian rarely wants *all* the data. Usually, you want specific subsets: "Show me all laborers from Cork."

In Pandas, we use **Boolean Indexing**.
* `df['Column'] == 'Value'` creates a list of True/False.
* `df[ ... ]` keeps only the True rows.

In [ ]:
# Filter 1: Show me only the passengers from 'Cork'
cork_passengers = df[df['Origin'] == 'Cork']
cork_passengers

We can also combine conditions using `&` (AND) and `|` (OR).

In [ ]:
# Filter 2: Show me 'Laborers' who are UNDER 25 years old
young_laborers = df[(df['Occupation'] == 'Laborer') & (df['Age'] < 25)]
young_laborers

## 6. Analysis: Grouping and Aggregating
This is the most powerful tool for the "Distant Reading" of history. Instead of reading row-by-row, we can aggregate data to find patterns.

**The `groupby` method:** This splits the data into groups, applies a math function (count, mean, sum), and combines the results.

**Question:** What were the most common occupations for this group?

In [ ]:
# Count the unique values in the 'Occupation' column
df['Occupation'].value_counts()

**Question:** What was the average age of passengers from each city?

In [ ]:
# Group by 'Origin', then calculate the Mean of 'Age'
avg_age_by_city = df.groupby('Origin')['Age'].mean()
print(avg_age_by_city)

## 7. Working with Time (Temporal Analysis)
Earlier, we saw that `Arrival Date` was a string. Let's convert it to a real DateTime object so we can ask questions like "How many people arrived per year?"

In [7]:
# Convert the column to datetime objects
df['Arrival Date'] = pd.to_datetime(df['Arrival Date'])

# Extract the 'Year' into a new column
df['Arrival Year'] = df['Arrival Date'].dt.year

df.head()

,PassengerID,Surname,Given Name,Age,Occupation,Origin,Arrival Date,Arrival Year
0,101,O'Malley,Patrick,24.0,Laborer,Cork,1845-04-12,1845
1,102,Schmidt,Hans,35.0,Baker,Hamburg,1848-06-20,1848
2,103,Romano,Giuseppe,19.0,Laborer,Naples,1849-11-02,1849
3,104,O'Malley,Mary,22.0,Seamstress,Cork,1845-04-12,1845
4,105,Dubois,Jean,40.0,Merchant,Lyon,1850-01-15,1850


Now we can see trends over time.

In [6]:
# Count arrivals by year
arrivals_per_year = df['Arrival Year'].value_counts().sort_index()
print(arrivals_per_year)

Arrival Year
1845    4
1847    1
1848    2
1849    1
1850    1
1852    1
1854    1
1855    1
Name: count, dtype: int64


## 8. Saving Your Results (The Deliverable)
You have filtered the data to just the "Young Laborers" and you want to save this subset to share with a colleague (or submit for your Milestone).

In [ ]:
# Save the filtered dataframe to a new CSV
young_laborers.to_csv('young_laborers_subset.csv', index=False)

print("File saved successfully.")

### Assignment for Today:
1.  Run the cells above.
2.  Modify the code to answer this question: **What is the average age of 'Bakers' in this dataset?**
3.  Filter the dataset to create a new file containing only passengers who arrived **after 1850**.